## 5. ETL: Conexión Python con MySQL Workbench

Partiendo de los archivos .csv limpios generados desde siguientes scripts de Python:
  1. ETL_mcc
  2. ETL_card
  3. ETL_transactions
  4. ETL_users

Los importaremos al presente script, y los conectaremos con MySQL Workbench con el fin de insertar los datos de los mismos en las tablas de MySQL Workbench que fueron creadas previamente en la base de datos 'moni_clean'(ver script sql: 'moni_clean_DDL').

### Librerías importadas

In [1]:
import numpy as np
import pandas as pd
import os
from pathlib import Path
import requests
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

### Constantes

* Directorio actual del script de Python

In [2]:
os.getcwd()

'C:\\Users\\germa\\Documents\\Python\\edicion_12.0\\Proyecto_integrador\\ETL_finales'

In [3]:
# os.chdir('C:/Users/germa/Documents/Python/edicion_12.0/Proyecto_integrador/ETL_finales')

In [4]:
# Consultamos el directorio actual de trabajo y lo guardamos en la constante 'CURRENT_DIRECTORY'
CURRENT_DIRECTORY=os.getcwd()
CURRENT_DIRECTORY

'C:\\Users\\germa\\Documents\\Python\\edicion_12.0\\Proyecto_integrador\\ETL_finales'

* Directorio de la carpeta 'files'

In [5]:
# Creamos una carpeta llamada 'files' dentro del directorio actual de trabajo
new_folder = 'files'

FILES_DIRECTORY=os.path.join(CURRENT_DIRECTORY, new_folder)

try:
    os.makedirs(FILES_DIRECTORY, exist_ok=True)
    print(f"Carpeta creada en: {FILES_DIRECTORY}")
    
except OSError as e:
    print(f"Error al crear la carpeta: {e}")

Carpeta creada en: C:\Users\germa\Documents\Python\edicion_12.0\Proyecto_integrador\ETL_finales\files


### Funciones creadas

In [6]:
# Carga las variables de entorno, crea la URL de conexión y crea el motor para establecer la conexión con la base de datos MySQL
def obtain_db_connection():
    # Importamos las credenciales desde el archivo .env
    load_dotenv()

    host = os.getenv('DB_HOST')
    port = os.getenv('DB_PORT')
    user = os.getenv('DB_USER')
    password = os.getenv('DB_PASSWORD')
    database = os.getenv('DB_NAME')

    # Verificación de variables obligatorias
    required_variable = ['DB_HOST', 'DB_PORT', 'DB_USER', 'DB_PASSWORD', 'DB_NAME']

    for var in required_variable:
        if not os.getenv(var):
            raise EnvironmentError(f"Variable de entorno requerida no encontrada: {var}")

    # Creamos la URL de conexión
    connection_url = f'mysql+pymysql://{user}:{password}@{host}:{port}/{database}'

    # Creamos el motor de sqlalchemy
    engine = create_engine(connection_url)
    
    # Devuelve el motor para poder realizar consultas SQL
    return engine

In [7]:
# Carga las variables de entorno, crea la URL de conexión y crea el motor para establecer la conexión con la base de datos MySQL 
# (para el caso de tablas con mucha cantidad de registros donde se deba usar la consulta SQL 'LOAD DATA LOCAL INFILE')
def obtain_db_connection_local_infile():
    # Importamos las credenciales desde el archivo .env
    load_dotenv()

    host = os.getenv('DB_HOST')
    port = os.getenv('DB_PORT')
    user = os.getenv('DB_USER')
    password = os.getenv('DB_PASSWORD')
    database = os.getenv('DB_NAME')

    # Verificación de variables obligatorias
    required_variable = ['DB_HOST', 'DB_PORT', 'DB_USER', 'DB_PASSWORD', 'DB_NAME']

    for var in required_variable:
        if not os.getenv(var):
            raise EnvironmentError(f"Variable de entorno requerida no encontrada: {var}")

    # Creamos la URL de conexión
    connection_url = f'mysql+pymysql://{user}:{password}@{host}:{port}/{database}'

    # Creamos el motor de sqlalchemy
    engine = create_engine(connection_url, connect_args={'local_infile': True})
    
    # Devuelve el motor para poder realizar consultas SQL
    return engine

### Cambio de directorio de trabajo a 'FILES_DIRECTORY'

* Cambiamos el directorio de trabajo a 'FILES_DIRECTORY' porque desde allí importaremos los archivos 'limpios' en formato .csv al presente script

In [8]:
os.chdir(FILES_DIRECTORY)

In [9]:
# Chequeamos el directorio actual de trabajo

In [10]:
pwd!

'C:\\Users\\germa\\Documents\\Python\\edicion_12.0\\Proyecto_integrador\\ETL_finales\\files'

### Importación de archivos .csv al presente script de Python

* Importamos los archivos 'limpios' en Python como dataframes

In [11]:
mcc_category = pd.read_csv('mcc_category.csv')
mcc = pd.read_csv('mcc.csv')
card_brand = pd.read_csv('card_brand.csv')
card_type = pd.read_csv('card_type.csv')
card = pd.read_csv('card.csv')
transaction_type = pd.read_csv('transaction_type.csv')
error_type = pd.read_csv('error_type.csv')
transactions = pd.read_csv('transactions.csv')
transactions_error_type = pd.read_csv('transactions_error_type.csv')
city = pd.read_csv('city.csv')
country = pd.read_csv('country.csv')
state_usa = pd.read_csv('state_usa.csv')
users = pd.read_csv('users.csv')

* Exploramos la información general de los dataframes 'limpios' a conectar con MySQL Workbench

In [12]:
dataframe_list = [mcc_category, card_brand, card_type, transaction_type, error_type, city, country, state_usa, mcc, users, card, transactions, transactions_error_type]
dataframe_name = ['mcc_category', 'card_brand', 'card_type', 'transaction_type', 'error_type', 'city', 'country', 'state_usa', 'mcc', 'users', 'card', 'transactions', 'transactions_error_type']

for i, df in enumerate(dataframe_list):
    print('\n', f'dataframe ** {dataframe_name[i]} **:')
    print(df.info(show_counts=True), '\n')


 dataframe ** mcc_category **:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id_mcc_category  12 non-null     int64 
 1   mcc_category     12 non-null     object
dtypes: int64(1), object(1)
memory usage: 324.0+ bytes
None 


 dataframe ** card_brand **:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id_card_brand  4 non-null      int64 
 1   card_brand     4 non-null      object
dtypes: int64(1), object(1)
memory usage: 196.0+ bytes
None 


 dataframe ** card_type **:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id_card_type  3 non-null      int64 
 1   

### Configuración de la conexión e inserción de los datos en la BBDD

* Para conectar los dataframes con las tablas de SQL se siguió el siguiente orden debido a las relaciones entre las tablas hechas anteriormente en el script DDL de SQL:

#### Tablas SQL con PK solamente

In [13]:
# Llamamos a la función 'obtain_db_connection' que carga las variables de entorno, crea la URL de conexión y crea el motor para 
# establecer la conexión con la base de datos MySQL
engine = obtain_db_connection()

# Establecemos la conexión con la BBDD
connection = engine.connect()

# Dataframes a partir de los cuales les vamos a insertar los datos a las tablas de la BBDD: Tablas SQL con PK solamente
dataframe_list = [mcc_category, card_brand, card_type, transaction_type, error_type, city, country, state_usa]
dataframe_name = ['mcc_category', 'card_brand', 'card_type', 'transaction_type', 'error_type', 'city', 'country', 'state_usa']

# Insertamos los datos a las tablas de la BBDD con los datos de los dataframes
for i, df in enumerate(dataframe_list):
    try:
        df.to_sql(name=dataframe_name[i], con=engine, if_exists='append', index=False);

    except ValueError as vx:
        print(f'Error: {vx}')
    except Exception as ex:   
        print(f'Error: {ex}')
    else:
        # Si no da error, imprime que se han insertado correctamente los datos en las tablas de la BBDD
        print(f'Los datos de la tabla {dataframe_name[i]} han sido insertados correctamente.');

# Cerramos la conexión con la BBDD
connection.close()

Los datos de la tabla mcc_category han sido insertados correctamente.
Los datos de la tabla card_brand han sido insertados correctamente.
Los datos de la tabla card_type han sido insertados correctamente.
Los datos de la tabla transaction_type han sido insertados correctamente.
Los datos de la tabla error_type han sido insertados correctamente.
Los datos de la tabla city han sido insertados correctamente.
Los datos de la tabla country han sido insertados correctamente.
Los datos de la tabla state_usa han sido insertados correctamente.


#### Tablas SQL con PK y FK

In [14]:
# Llamamos a la función 'obtain_db_connection' que carga las variables de entorno, crea la URL de conexión y crea el motor para 
# establecer la conexión con la base de datos MySQL
engine = obtain_db_connection()

# Establecemos la conexión con la BBDD
connection = engine.connect()

# Dataframes a partir de los cuales les vamos a insertar los datos a las tablas de la BBDD: Tablas SQL con PK solamente
dataframe_list = [mcc, users]
dataframe_name = ['mcc', 'users']

# Insertamos los datos a las tablas de la BBDD con los datos de los dataframes
for i, df in enumerate(dataframe_list):
    try:
        df.to_sql(name=dataframe_name[i], con=engine, if_exists='append', index=False);

    except ValueError as vx:
        print(f'Error: {vx}')
    except Exception as ex:   
        print(f'Error: {ex}')
    else:
        # Si no da error, imprime que se han insertado correctamente los datos en las tablas de la BBDD
        print(f'Los datos de la tabla {dataframe_name[i]} han sido insertados correctamente.');

# Cerramos la conexión con la BBDD
connection.close()

Los datos de la tabla mcc han sido insertados correctamente.
Los datos de la tabla users han sido insertados correctamente.


In [12]:
# Llamamos a la función 'obtain_db_connection' que carga las variables de entorno, crea la URL de conexión y crea el motor para 
# establecer la conexión con la base de datos MySQL
engine = obtain_db_connection()

# Establecemos la conexión con la BBDD
connection = engine.connect()

# Dataframes a partir de los cuales les vamos a insertar los datos a las tablas de la BBDD: Tablas SQL con PK solamente
dataframe_list = [card]
dataframe_name = ['card']

# Insertamos los datos a las tablas de la BBDD con los datos de los dataframes
for i, df in enumerate(dataframe_list):
    try:
        df.to_sql(name=dataframe_name[i], con=engine, if_exists='append', index=False);

    except ValueError as vx:
        print(f'Error: {vx}')
    except Exception as ex:   
        print(f'Error: {ex}')
    else:
        # Si no da error, imprime que se han insertado correctamente los datos en las tablas de la BBDD
        print(f'Los datos de la tabla {dataframe_name[i]} han sido insertados correctamente.');

# Cerramos la conexión con la BBDD
connection.close()

Los datos de la tabla card han sido insertados correctamente.


##### Dataframe 'transactions'

In [13]:
# Como el dataframe 'transactions' contiene millones de registros, vamos a insertar sus datos a la tabla de SQL de esta manera:
# 1) Consultamos cuál es el directorio seguro de la PC donde MySQL va a leer el archivo en formato .csv 
# 'transactions.csv'. Vamos a hacer una llamada a la BBDD para averiguar cuál es el directorio seguro de la PC para MySQL:

# Llamamos a la función 'obtain_db_connection' que carga las variables de entorno, crea la URL de conexión y crea el motor para 
# establecer la conexión con la base de datos MySQL
engine = obtain_db_connection()

# Establecemos la conexión con la BBDD
connection = engine.connect()

try:
    # Creamos la conexión
    with connection:
        
        # Usamos la conexión para solicitar el nombre de la ruta que MySQL necesita que esté el archivo csv allí para leerlo
        result = connection.execute(text("SHOW VARIABLES LIKE 'secure_file_priv';"))
        
        # Imprimimos los resultados de la consulta
        path = result.fetchone()[1]
        file_secure_path_sql = Path(path)
        print('El directorio seguro para MySQL es: ', path)
        
except Exception as e:
    print(f'Error: {e}')

El directorio seguro para MySQL es:  C:\ProgramData\MySQL\MySQL Server 9.5\Uploads\


In [14]:
# 2) Cambiamos el directorio actual de trabajo al directorio seguro para MySQL para subir el archivo csv: 'transactions.csv'
os.chdir(file_secure_path_sql)

In [15]:
# Chequeamos el directorio actual de trabajo

In [16]:
pwd!

'C:\\ProgramData\\MySQL\\MySQL Server 9.5\\Uploads'

In [17]:
# 3) Exportamos el archivo 'transactions.csv' al directorio seguro de MySQL
transactions.to_csv('transactions.csv', index=False)

In [18]:
# Llamamos a la función 'obtain_db_connection_local_infile' que carga las variables de entorno, crea la URL de conexión y crea el motor para 
# establecer la conexión con la base de datos MySQL
engine_local_infile = obtain_db_connection_local_infile()

# Establecemos la conexión con la BBDD
connection = engine_local_infile.connect()
connection_raw = engine_local_infile.raw_connection()
cursor = connection_raw.cursor()

# 6) Dataframe a partir del cual le vamos a insertar los datos a la tabla de la BBDD: Tabla SQL con PK y FK
file_secure_path = (file_secure_path_sql / 'transactions.csv').as_posix()
table_name = 'transactions'

# 7) Construimos la sentencia SQL
sql = f"""
LOAD DATA LOCAL INFILE '{file_secure_path}'
INTO TABLE {table_name}
FIELDS TERMINATED BY ',' 
ENCLOSED BY '"'
LINES TERMINATED BY '\n'
IGNORE 1 ROWS;
"""

# 8) Habilitamos el servidor para poder utilizar la sentencia SQL llamada 'LOAD DATA LOCAL INFILE', luego chequeamos que esté en 
# estado 'ON'
try:   
    connection.execute(text("SET GLOBAL local_infile = 1;"))
    
    # Verificamos el cambio
    result = connection.execute(text("SHOW VARIABLES LIKE 'local_infile';"))
    print(result.fetchone())

except ValueError as vx:
    print(f'Error: {vx}')
except Exception as ex:   
    print(f'Error: {ex}')

else:
     # Si no da error, imprime que se ha habilitado el servidor
    print("local_infile habilitado globalmente.")

# 9) Insertamos los datos a la tabla de la BBDD con los datos del dataframe
try:
    cursor.execute(sql)
    connection_raw.commit()

except ValueError as vx:
    print(f'Error: {vx}')
except Exception as ex:   
    print(f'Error: {ex}')

else:
    # Si no da error, imprime que se han insertado correctamente los datos en la tabla de la BBDD
    print(f'Los datos de la tabla {table_name} han sido insertados correctamente.');

# 10) Cerramos la conexión con la BBDD
connection.close()
connection_raw.close()

('local_infile', 'ON')
local_infile habilitado globalmente.
Los datos de la tabla transactions han sido insertados correctamente.


#### Tabla SQL con FK solamente

In [19]:
# Llamamos a la función 'obtain_db_connection' que carga las variables de entorno, crea la URL de conexión y crea el motor para
# establecer la conexión con la base de datos MySQL
engine = obtain_db_connection()

# Establecemos la conexión con la BBDD
connection = engine.connect()

# Dataframe a partir del cual le vamos a insertar los datos a la tabla de la BBDD: Tabla SQL con FK solamente
dataframe = transactions_error_type
table_name = 'transactions_error_type'

# Insertamos los datos a la tabla de la BBDD con los datos del dataframe
try:
    dataframe.to_sql(name=table_name, con=engine, if_exists='append', index=False);

except ValueError as vx:
    print(f'Error: {vx}')
except Exception as ex:   
    print(f'Error: {ex}')

else:
    # Si no da error, imprime que se han insertado correctamente los datos en la tabla de la BBDD
    print(f'Los datos de la tabla {table_name} han sido insertados correctamente.');

# Cerramos la conexión con la BBDD
connection.close()

Los datos de la tabla transactions_error_type han sido insertados correctamente.


### Consultas SQL para chequear la inserción de datos en la BBDD

In [20]:
# Llamamos a la función 'obtain_db_connection' que carga las variables de entorno, crea la URL de conexión y crea el motor para
# establecer la conexión con la base de datos MySQL
engine = obtain_db_connection()

# Establecemos la conexión con la BBDD
connection = engine.connect()

# Dataframes que ya fueron insertados sus datos en las tablas de la BBDD
dataframe_list = [mcc_category, card_brand, card_type, transaction_type, error_type, city, country, state_usa, mcc, users, card, transactions, transactions_error_type]
dataframe_name = ['mcc_category', 'card_brand', 'card_type', 'transaction_type', 'error_type', 'city', 'country', 'state_usa', 'mcc', 'users', 'card', 'transactions', 'transactions_error_type']

# Hacemos una consultas por tabla a la BBDD para ver las primeras 10 filas de cada tabla
for i, df in enumerate(dataframe_list):

    try:
        result = pd.read_sql(f"SELECT * FROM {dataframe_name[i]} LIMIT 10;", engine)
        print('\n**', dataframe_name[i], '**',':', '\n', result)
        
    except ValueError as vx:
        print(f'Error: {vx}')
    except Exception as ex:   
        print(f'Error: {ex}')
    
    else:
        # Si no da error, imprime que la consulta a la BBDD se ha realizado correctamente
        print(f'La consulta a la tabla "{dataframe_name[i]}" ha sido realizada correctamente.'); 

# Cerramos la conexión con la BBDD
connection.close()


** mcc_category ** : 
    id_mcc_category                mcc_category
0                1     digital & subscriptions
1                2  entertainment & recreation
2                3          financial services
3                4             food & beverage
4                5                  healthcare
5                6  industrial & manufacturing
6                7           personal services
7                8       professional services
8                9           retail & shopping
9               10              transportation
La consulta a la tabla "mcc_category" ha sido realizada correctamente.

** card_brand ** : 
    id_card_brand  card_brand
0              1        Amex
1              2    Discover
2              3  Mastercard
3              4        Visa
La consulta a la tabla "card_brand" ha sido realizada correctamente.

** card_type ** : 
    id_card_type      card_type
0             1         credit
1             2          debit
2             3  debit_prepaid
La cons